<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的配套代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库： <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第4章：习题解答

本 Notebook 中用到的依赖包括：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # 供 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

&nbsp;
# 练习 4.1：在 MATH-500 上使用链式思维提示 (Chain-of-Thought Prompting)。

- 只需要在应用提示模板之后额外拼接一句后缀即可，例如 `"\n\nExplain step by step."` 
- 下方展示了在第 3 章基础上修改过的 MATH-500 评估函数（通过 `# NEW` 注释标出改动处）

```python
import json                                                # 处理 JSON 数据
from pathlib import Path                                   # 跨平台文件路径
import time                                                 # 计时

from reasoning_from_scratch.ch03 import (                  # 导入评测所需工具
    eta_progress_message,                                  # 进度与 ETA 打印
    extract_final_candidate,                               # 提取最终答案
    render_prompt,                                         # 构造提示词
    grade_answer,                                          # 判分函数
    generate_text_stream_concat,                           # 流式生成文本
)

def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=512,
    verbose=False,
    prompt_suffix=""  # NEW：允许在提示词后追加自定义后缀
):
    if out_path is None:
        dev_name = str(device).replace(":", "-")           # 设备名做文件名（兼容 Windows）
        out_path = Path(f"math500-{dev_name}.jsonl")       # 默认输出路径

    num_examples = len(math_data)                          # 测试样本总数
    num_correct = 0                                        # 累计正确数
    start_time = time.time()                               # 开始计时

    with open(out_path, "w", encoding="utf-8") as f:       # 写入日志文件
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"])         # 构造提示词
            prompt += prompt_suffix                        # NEW：附加提示后缀（如链式思考指令）
            gen_text = generate_text_stream_concat(        # 生成模型回答
                model, tokenizer, prompt, device,
                max_new_tokens=max_new_tokens,
                verbose=verbose,
            )

            extracted = extract_final_candidate(gen_text)  # 提取最终答案
            is_correct = grade_answer(                     # 判分
                extracted, row["answer"]
            )
            num_correct += int(is_correct)                 # 累计正确数

            record = {                                     # 构造日志记录
                "index": i,
                "problem": row["problem"],
                "gtruth_answer": row["answer"],
                "generated_text": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")  # 逐行写入 JSON

            progress_msg = eta_progress_message(           # 打印进度和 ETA
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500",
            )
            print(progress_msg, end="\r", flush=True)
            if verbose:                                    # 需要详细日志则额外打印
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"Correct so far: {num_correct}\n{'-'*50}"
                )

    seconds_elapsed = time.time() - start_time             # 总耗时
    acc = num_correct / num_examples if num_examples else 0.0  # 准确率
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")
    print(f"Total time: {seconds_elapsed/60:.1f} min")
    print(f"Logs written to: {out_path}")
    return num_correct, num_examples, acc                  # 返回正确数、总数和准确率

```

- 下表给出了相比第 3 章基线的性能提升。

|    | 方法                                       | 模型     | 准确率 | 耗时       |
|----|----------------------------------------------|-----------|----------|------------|
| 1  | 基线（第 3 章），贪婪解码        | 基础      | 15.2%    | 10.1 分钟   |
| 2  | 基线（第 3 章），贪婪解码        | 推理 | 48.2%    | 182.1 分钟  |
| 3  | 思维链提示（"CoT"）           | 基础      | 40.6%    | 84.5 分钟   |

- 为方便起见，你可以直接运行 [cot_prompting_math500.py](../02_math500-inference-scaling-scripts/cot_prompting_math500.py) 目录下的 [../02_math500-inference-scaling-scripts](../02_math500-inference-scaling-scripts)

&nbsp;
## 练习 4.2：在 MATH-500 上尝试温度缩放与 top-p 过滤。     

- 在这里需要把 `generate_text_stream_concat` 替换成 `generate_text_stream_concat_flex`，并将 `generate_text_top_p_stream_cache` 注入进去。
- 下方展示了第 3 章的 MATH-500 评估函数在此基础上的改动（以 `# NEW` 注释标示）。

```python
import json                                 # 处理 JSON 数据
from pathlib import Path                    # 跨平台路径处理
import time                                 # 计时

from reasoning_from_scratch.ch03 import (   # 导入第 3 章的工具函数
    eta_progress_message,                   # 进度与 ETA 显示
    extract_final_candidate,                # 提取最终答案
    render_prompt,                          # 构造提示词
    grade_answer,                           # 判分函数
    generate_text_stream_concat,            # 基础流式生成（未使用）
)
from reasoning_from_scratch.ch04 import generate_text_stream_concat_flex  # 可插拔生成包装器

def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=512,
    verbose=False,
    temperature=1.0,  # NEW 温度采样参数
    top_p=1.0,        # NEW top-p 过滤阈值
):
    if out_path is None:                                     # 若未指定输出路径则用默认
        dev_name = str(device).replace(":", "-")             # 设备名中冒号替换为连字符
        out_path = Path(f"math500-{dev_name}.jsonl")         # 默认输出文件名

    num_examples = len(math_data)                            # 样本总数
    num_correct = 0                                          # 累计正确数
    start_time = time.time()                                 # 计时起点

    with open(out_path, "w", encoding="utf-8") as f:         # 打开日志输出文件
        for i, row in enumerate(math_data, start=1):         # 逐题处理
            prompt = render_prompt(row["problem"])           # 构造提示词
            gen_text = generate_text_stream_concat_flex(     # NEW 使用可插拔生成器
                model, tokenizer, prompt, device,
                max_new_tokens=max_new_tokens,
                verbose=verbose,
                generate_func=generate_text_top_p_stream_cache,  # NEW 指定带 top-p/温度的生成函数
                temperature=temperature,                         # NEW 传入温度
                top_p=top_p                                      # NEW 传入 top-p
            )

            extracted = extract_final_candidate(gen_text)    # 提取最终答案
            is_correct = grade_answer(extracted, row["answer"])  # 判分
            num_correct += int(is_correct)                   # 累计正确数

            record = {                                       # 记录当前题目的详细日志
                "index": i,
                "problem": row["problem"],
                "gtruth_answer": row["answer"],
                "generated_text": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")  # 写入一行 JSON

            progress_msg = eta_progress_message(             # 构造进度与 ETA 信息
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500",
            )
            print(progress_msg, end="\r", flush=True)        # 单行刷新显示进度
            if verbose:                                      # 若需详细日志则额外打印
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"Correct so far: {num_correct}\n{'-'*50}"
                )

    seconds_elapsed = time.time() - start_time               # 总耗时
    acc = num_correct / num_examples if num_examples else 0.0  # 计算准确率
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")  # 打印准确率
    print(f"Total time: {seconds_elapsed/60:.1f} min")       # 打印耗时
    print(f"Logs written to: {out_path}")                    # 输出日志文件位置
    return num_correct, num_examples, acc                    # 返回结果元组

```

- 当 `temperature`=0.9、`top_p`=0.9 时，和基线（下表第 1 行）相比仅有细微差异，这是预期现象，因为这套配置只是为后续的自洽采样做准备。

|      | 方法                                    | 模型     | 准确率 | 耗时      |
| ---- | ----------------------------------------- | --------- | -------- | --------- |
| 1    | 基线（第 3 章），贪婪解码     | 基础      | 15.2%    | 10.1 分钟  |
| ...  | ...                                       | ...       | ...      | ...       |
| 4    | 温度与 top-p（"Top-p"）           | 基础      | 17.8%    | 30.7 分钟  |

- 为了方便，你可以直接运行 [self_consistency_math500.py](../02_math500-inference-scaling-scripts/self_consistency_math500.py) 下的 [../02_math500-inference-scaling-scripts](../02_math500-inference-scaling-scripts)
- 虽然它本质是自洽采样脚本，但若设置`--num_samples 1`, 就等同于禁用自洽流程。

&nbsp;
## 练习 4.3：在 MATH-500 上进行自洽采样

- 以第 3 章的 `evaluate_math500_stream `为基准，第一项修改是把 `gen_text = generate_text_stream_concat(...) `替换成第 4 章的 `results = self_consistency_vote(...)`。
- 第二项修改是实现简单的平票处理：如果出现多个“最高频”答案，就选它们在 `results["majority_winners"]` 中的第一个元素（例如结果集为 1, 3, 5, 3, 5 时，返回 3）。
- 因此，拿到 `results["majority_winners"]` 后，只需取 `results["majority_winners"][0]` 作为最终答案就能在平票情况下做裁决。

```python
import json                                                # 读取/写入 JSON 数据
from pathlib import Path                                   # 跨平台路径处理
import time                                                # 计时

from reasoning_from_scratch.ch03 import (                  # 导入评测所需工具
    eta_progress_message,                                  # 打印进度与 ETA
    render_prompt,                                         # 构造提示词
    grade_answer,                                          # 判分函数
)
from reasoning_from_scratch.ch04 import self_consistency_vote  # NEW 导入自一致性投票生成

def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",    # NEW 允许在提示后追加后缀
    temperature=1.0,     # NEW 温度采样参数
    top_p=1.0,           # NEW top-p 过滤阈值
    seed=None,           # NEW 随机种子（保证可复现）
    num_samples=10,      # NEW 每题采样次数
):
    if out_path is None:                                   # 未指定输出路径则构造默认文件名
        dev_name = str(device).replace(":", "-")           # 设备名中的冒号替换为连字符
        out_path = Path(f"math500-{dev_name}.jsonl")       # 默认输出文件

    num_examples = len(math_data)                          # 样本总数
    num_correct = 0                                        # 累计正确数
    start_time = time.time()                               # 开始计时

    with open(out_path, "w", encoding="utf-8") as f:       # 打开输出文件
        for i, row in enumerate(math_data, start=1):       # 逐条处理题目
            prompt = render_prompt(row["problem"])         # 构造基础提示词

            ##############################################################
            # NEW 使用自一致性投票生成并取最终答案
            prompt += prompt_suffix                        # NEW 追加提示后缀（如链式思考指令）
            results = self_consistency_vote(               # NEW 多次采样并投票
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            if results["final_answer"] is None:            # NEW 并列时取第一个多数答案
                extracted = results["majority_winners"][0]
            else:
                extracted = results["final_answer"]

            # 找到与最终短答案对应的完整回答
            if extracted is not None:
                for idx, s in enumerate(results["short_answers"]):
                    if s == extracted:
                        long_answer = results["full_answers"][idx]
                        break
            gen_text = long_answer                         # 记下对应的完整回答
            ##############################################################

            is_correct = grade_answer(                     # 判分：预测 vs 标准
                extracted, row["answer"]
            )
            num_correct += int(is_correct)                 # 累计正确数量

            record = {                                     # 记录当前题目的详细信息
                "index": i,
                "problem": row["problem"],
                "gtruth_answer": row["answer"],
                "generated_text": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")  # 写入一行 JSON

            progress_msg = eta_progress_message(           # 生成进度消息
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500",
            )
            print(progress_msg, end="\r", flush=True)      # 单行刷新显示进度
            if verbose:                                    # 若需详细日志则额外打印
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"Correct so far: {num_correct}\n{'-'*50}"
                )

    seconds_elapsed = time.time() - start_time             # 计算耗时
    acc = num_correct / num_examples if num_examples else 0.0  # 计算准确率
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")  # 输出准确率
    print(f"Total time: {seconds_elapsed/60:.1f} min")     # 输出耗时
    print(f"Logs written to: {out_path}")                  # 输出日志文件位置
    return num_correct, num_examples, acc                  # 返回结果

```

- 自洽采样带来的性能提升汇总见下表（第 5–7 行与第 9–12 行）。

|      | 方法                                    | 模型     | 准确率 | 耗时      |
| ---- | ----------------------------------------- | --------- | -------- | --------- |
| 1    | 基线（第 3 章），贪婪解码     | 基础      | 15.2%    | 10.1 分钟  |
| 2    | 基线（第 3 章），贪婪解码     | 推理 | 48.2%    | 182.1 分钟 |
| 3    | 思维链提示（"CoT"）        | 基础      | 40.6%    | 84.5 分钟  |
| 4    | 温度与 top-p（"Top-p"）           | 基础      | 17.8%    | 30.7 分钟  |
| 5    | "Top-p" + 自洽性 (n=3)          | 基础      | 29.6%    | 97.6 分钟  |
| 6    | "Top-p" + 自洽性 (n=5)          | 基础      | 27.8%    | 116.8 分钟 |
| 7    | "Top-p" + 自洽性 (n=10)         | 基础      | 31.6%    | 300.4 分钟 |
| 8    | "Top-p" + "CoT"                           | 基础      | 33.4%    | 129.2 分钟 |
| 9    | 自洽性 (n=3) + "Top-p" + "CoT"  | 基础      | 42.2%    | 211.6 分钟 |
| 10   | 自洽性 (n=5) + "Top-p" + "CoT"  | 基础      | 48.0%    | 452.9 分钟 |
| 11   | 自洽性 (n=10) + "Top-p" + "CoT" | 基础      | 52.0%    | 862.6 分钟 |
| 12   | 自洽性 (n=3) + "Top-p" + "CoT"  | 推理 | 55.2%    | 544.4 分钟 |

- 为方便复现这些结果，可以运行 [../02_math500-inference-scaling-scripts](../02_math500-inference-scaling-scripts) 目录中的 [self_consistency_math500.py](../02_math500-inference-scaling-scripts/self_consistency_math500.py) 脚本；该目录还说明了应使用哪些设置。


&nbsp;
## 练习 4.4：在自洽采样中加入早停机制。

- 早停判定只需在代码里增加几行检查逻辑：如果某个候选答案已被统计多次——更精确地说，当它的出现次数大于 num_samples / 2 时，就可以提前终止采样。

```python
if early_stop and counts[short] > num_samples / 2:  # 若开启提前停止且某答案超过半数票
    majority_winners = [short]                      # 记录当前答案为唯一多数
    final_answer = short                            # 将其设为最终答案
    break                                           # 立即结束循环

```

- 完整的修改后函数如下所示，所有改动都用 `# New` 标注。

```python
import torch                                             # 引入 PyTorch，用于张量与采样
from collections import Counter                          # 计数器，用于统计答案频次

from reasoning_from_scratch.ch03 import (                # 导入提取最终答案的工具
    extract_final_candidate,
)
from reasoning_from_scratch.ch04 import (                # 导入可插拔生成接口与带 top-p 的生成器
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache,
)

def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,                                      # 默认采样次数
    temperature=0.8,                                     # 采样温度
    top_p=0.9,                                           # top-p 阈值
    max_new_tokens=2048,                                 # 生成的最大 token 数
    show_progress=True,                                  # 是否打印进度
    show_long_answer=False,                              # 是否打印完整回答
    seed=None,                                           # 随机种子，保证可复现
    early_stop=True,   # NEW 提前停止开关（达到多数即停）
):
    full_answers, short_answers = [], []                 # 保存完整回答与提取后的短答案
    counts = Counter()                                   # 统计各短答案出现次数
    groups = {}                                          # 记录每个短答案对应的样本索引列表
    majority_winners, final_answer = [], None            # 多数票答案列表与最终答案初始化

    for i in range(num_samples):                         # 循环多次采样
        if seed is not None:                             # 如设置了种子，则为本次采样设定确定性随机性
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(       # 生成完整回答（可插拔生成器）
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(                 # 提取最终短答案
            answer, fallback="number_then_full"
        )
        full_answers.append(answer)                      # 记录完整回答
        short_answers.append(short)                      # 记录短答案
        counts[short] += 1                               # 更新计数
        groups.setdefault(short, []).append(i)           # 记录该答案对应的样本索引

        if show_progress:                                # 如需打印进度则输出本次短答案
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

        #########################################################
        # NEW 提前停止：若某答案票数已超过 50%，立即结束
        if early_stop and counts[short] > num_samples / 2:
            majority_winners = [short]                   # 记录唯一多数答案
            final_answer = short                         # 设定最终答案
            break                                        # 跳出采样循环
        #########################################################

    if final_answer is None:                             # 若未提前确定最终答案，则按频次统计
        mc = counts.most_common()                        # 按出现频次排序
        if mc:
            top_freq = mc[0][1]                          # 最高频次
            majority_winners = [s for s, f in mc if f == top_freq]  # 可能存在并列
            final_answer = mc[0][0] if len(majority_winners) == 1 else None  # 唯一多数则确定最终答案

    return {                                             # 返回所有结果
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }

```

- 若要在 MATH-500 数据集上应用这个修改版函数，可在 [../02_math500-inference-scaling-scripts](../02_math500-inference-scaling-scripts) 目录下运行[self_consistency_math500.py](../02_math500-inference-scaling-scripts/self_consistency_math500.py)，并加上 `--early_stop` 参数。